# Translation-command dropout: Kaggle K=1 cohort

**60 episodes:** three LIBERO tasks × ten initial states × two probabilities (`p=0.1, 0.3`).
Every control step replans (`K=1`). With probability `p`, only the clipped XYZ command is replaced by zero; rotation and gripper commands are preserved. The same deterministic disturbance stream is used on the home machine.

In Kaggle **Settings**, enable a GPU accelerator and Internet, then run all cells. One GPU is used. This notebook embeds all experiment code and compact reference metadata; no repository clone, extra dataset, SSH connection or credentials are needed. Public model weights and simulator assets are downloaded at their frozen revisions. The isolated environment and model cache live under `/tmp`; only results are written under `/kaggle/working`.

The main home collection runs the other eight methods. Existing `p=0` results are reused; **this notebook does not add 30 baseline episodes**. Three short excluded checks establish that the dropout wrapper is exactly transparent at `p=0` on the Kaggle GPU. Initial-state bytes must match home. GPU-dependent first observations/actions are logged, and the two probabilities must have exactly matching local initial observations and first predictions. Cross-GPU success/call comparisons carry a hardware limitation; wall-clock timings must remain separate.

This notebook was generated from the frozen project sources. GPU execution must pass its checks on Kaggle before its results are usable. Installation/download time depends on Kaggle networking; no fixed completion time is promised.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, subprocess, sys, zipfile, io

EMBEDDED_BUNDLE = '...'
BUNDLE_BYTES = base64.b64decode(EMBEDDED_BUNDLE)
# A revised bundle gets a fresh setup directory; previous results stay intact.
BUNDLE_ID = hashlib.sha256(BUNDLE_BYTES).hexdigest()[:12]
ROOT = Path(f"/tmp/vla-dropout-kaggle-{BUNDLE_ID}")
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(BUNDLE_BYTES)) as archive:
    for entry in archive.infolist():
        target = (ROOT / entry.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()) or entry.filename.startswith("/"):
            raise RuntimeError("Unsafe embedded archive path")
        if target.exists() and target.is_file() and target.read_bytes() != archive.read(entry):
            raise RuntimeError(f"Existing source differs: {target}; choose a fresh ROOT")
    archive.extractall(ROOT)
manifest = json.loads((ROOT / "bundle-manifest.json").read_text())
for name, expected in manifest["files"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, name
os.environ["HF_HOME"] = "/tmp/vla-dropout-hf-cache"
os.environ["PIP_CACHE_DIR"] = "/tmp/vla-dropout-pip-cache"
os.environ["UV_CACHE_DIR"] = "/tmp/vla-dropout-uv-cache"
print("Verified embedded bundle:", len(manifest["files"]), "files")
print("Results will be saved under /kaggle/working")


## Install the frozen environment and download public assets

This installs the pinned standalone uv binary, then uses an isolated managed Python 3.12.3 and the exact project dependency lock. It does not require the notebook kernel's pip or ensurepip. Keep Internet enabled for this cell. Subsequent collection uses offline caches. If Kaggle reports an unavailable package or incompatible CUDA driver, stop and retain the error; do not silently replace package versions. See the [official uv installer documentation](https://docs.astral.sh/uv/reference/installer/).


In [ ]:
subprocess.run(["bash", str(ROOT / "scripts/setup_action_dropout_kaggle.sh")], cwd=ROOT, check=True)
PYTHON = ROOT / ".venv/bin/python"
print("Environment ready:", PYTHON)


## Optional resume

For a fresh run leave `RESUME_ZIP` empty. To resume a saved run, attach the previously downloaded `action-dropout-kaggle-results.zip` as a Kaggle input and set its path below. The collector verifies every completed episode and requires the same GPU type, package versions, configuration and source bundle. A different GPU/runtime starts a separate cohort and cannot silently resume these results. Completed episodes in the current session are reused automatically.


In [ ]:
RESUME_ZIP = ""  # e.g. /kaggle/input/my-dropout-checkpoint/action-dropout-kaggle-results.zip
OUTPUT = Path("/kaggle/working/action-dropout-kaggle")
ARCHIVE = Path("/kaggle/working/action-dropout-kaggle-results.zip")
if RESUME_ZIP:
    if OUTPUT.exists():
        raise RuntimeError("Output already exists; do not overwrite a current run with a resume archive")
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        for entry in archive.infolist():
            target = (Path("/kaggle/working") / entry.filename).resolve()
            if not target.is_relative_to(OUTPUT.resolve()) or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError("Unsafe resume archive path or symbolic link")
        archive.extractall("/kaggle/working")
    print("Resume files extracted; all evidence will be verified before reuse.")


## Run the 60 episodes

`MAX_HOURS` is a soft time limit checked between episodes, excluding setup and the short invariance checks. Set it to `0` to disable the limit. Leave `EPISODE_LIMIT=None` for all 60; a small positive value can check the pipeline and is resumed by rerunning this cell. The progress table includes only initial states completed at **both** probabilities. Each completed episode is committed atomically, and a downloadable ZIP is refreshed every five episodes and on normal completion/interruption. A hard Kaggle session kill can interrupt ZIP creation; the last completed ZIP and committed episode directory remain the checkpoints.

Kaggle **Save Version → Save & Run All** runs a separate session; it is not a live checkpoint of the current Python process. Download the ZIP from the running session or save the completed notebook outputs.


In [ ]:
MAX_HOURS = 10.0
EPISODE_LIMIT = None
command = [str(PYTHON), "-u", str(ROOT / "scripts/run_action_dropout_kaggle.py"),
           "--output", str(OUTPUT), "--archive", str(ARCHIVE), "--max-hours", str(MAX_HOURS)]
if EPISODE_LIMIT is not None:
    command += ["--limit", str(EPISODE_LIMIT)]
process = subprocess.Popen(command, cwd=ROOT)
try:
    code = process.wait()
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    process.wait()  # allow atomic export of completed episodes
    raise
if code:
    raise RuntimeError(f"Collector stopped with exit code {code}; inspect the error above and attempts/error.json")


## Verify and download results

Send the resulting ZIP back for joint analysis with the home collection. It contains raw commanded/applied actions, masks, model chunks, videos, reset/model references, the p=0 check and hardware provenance. Incomplete runs are explicitly identified; missing episodes are never counted as failures. The small table below is descriptive and does not establish cross-hardware causal equivalence.


In [ ]:
subprocess.run([str(PYTHON), str(ROOT / "scripts/run_action_dropout_kaggle.py"),
                "--output", str(OUTPUT), "--archive", str(ARCHIVE), "--verify-only"], cwd=ROOT, check=True)
from IPython.display import display, FileLink
progress = json.loads((OUTPUT / "progress.json").read_text())
print(f"Completed: {progress['completed_episodes']}/60; paired starts: {progress['complete_matched_starts']}/30")
for row in progress["conditions"]:
    print(f"p={row['p']:.1f}: {row['successes']}/{row['episodes']} success; "
          f"mean calls={row['mean_model_calls']:.1f}; realized dropout={row['drop_fraction']:.3f}")
display(FileLink(str(ARCHIVE)))
